# 02 — Scaled Dot-Product Attention
### Starter Notebook

**Learning objectives**
- Understand the Query / Key / Value abstraction
- Implement scaled dot-product attention from scratch
- Understand why scaling by $\sqrt{d_k}$ matters
- Visualise attention weights on real text
- Implement causal masking for autoregressive models

**Exercises** are marked `# TODO`.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Part A — The Q/K/V Intuition

Think of attention like a soft dictionary lookup:
- **Query (Q)**: what am I looking for?
- **Key (K)**: what does each position advertise?
- **Value (V)**: what information does each position contain?

The similarity between a query and each key determines how much of each value we retrieve.

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

### Exercise A1 — Implement `scaled_dot_product_attention`

Complete the function below without importing from `src.models.attention`.

In [ ]:
import math
from typing import Optional, Tuple


def my_attention(
    query: torch.Tensor,
    key:   torch.Tensor,
    value: torch.Tensor,
    mask:  Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Scaled dot-product attention.

    Args:
        query : (batch, n_heads, seq_q, d_k)
        key   : (batch, n_heads, seq_k, d_k)
        value : (batch, n_heads, seq_k, d_v)
        mask  : optional bool/float mask — 0 where we should NOT attend

    Returns:
        output          : (batch, n_heads, seq_q, d_v)
        attention_weights: (batch, n_heads, seq_q, seq_k)
    """
    d_k = query.size(-1)

    # TODO 1: compute raw attention scores  →  shape (batch, n_heads, seq_q, seq_k)
    scores = None

    # TODO 2: apply mask (set masked positions to -inf so softmax ignores them)
    if mask is not None:
        pass

    # TODO 3: softmax over the last dimension
    attn_weights = None

    # TODO 4: weighted sum of values
    output = None

    return output, attn_weights


# --- Quick correctness test ---
B, H, S, dk = 2, 4, 6, 16
q = torch.randn(B, H, S, dk)
k = torch.randn(B, H, S, dk)
v = torch.randn(B, H, S, dk)

out, w = my_attention(q, k, v)
if out is not None:
    print(f'Output shape  : {out.shape}   # expect {(B, H, S, dk)}')
    print(f'Weights shape : {w.shape}    # expect {(B, H, S, S)}')
    print(f'Weights sum   : {w.sum(-1).mean():.4f}  # should be 1.0')
else:
    print('Not yet implemented.')

### Exercise A2 — Why $\sqrt{d_k}$?

Without scaling, dot-products grow with $d_k$ and push softmax into saturated regions where gradients vanish.

Plot the softmax output entropy as a function of $d_k$ with and without scaling.

In [ ]:
def softmax_entropy(logits: torch.Tensor) -> float:
    """Mean entropy (in bits) of softmax distribution over last dim."""
    probs = F.softmax(logits, dim=-1)
    log_probs = torch.log2(probs + 1e-9)
    return -(probs * log_probs).sum(-1).mean().item()


dims   = [8, 16, 32, 64, 128, 256, 512]
entropies_unscaled = []
entropies_scaled   = []

for d in dims:
    # random Q, K vectors for a single head
    q = torch.randn(64, d)   # seq=64
    k = torch.randn(64, d)
    scores = q @ k.T          # (64, 64)

    entropies_unscaled.append(softmax_entropy(scores))
    entropies_scaled.append(softmax_entropy(scores / math.sqrt(d)))

# TODO: plot both curves and observe the difference
# Hint: lower entropy → more peaked → effectively attending to fewer tokens
print('[Exercise A2] Implement and plot the two entropy curves.')

## Part B — Causal Masking

For autoregressive (left-to-right) generation, position $t$ must only attend to positions $\leq t$.

### Exercise B1 — Build a causal mask

In [ ]:
def make_causal_mask(seq_len: int, device=None) -> torch.Tensor:
    """
    Returns a lower-triangular mask of shape (1, 1, seq_len, seq_len).
    1 = position can attend, 0 = masked.
    """
    # TODO: create the mask using torch.tril
    pass


mask = make_causal_mask(6)
if mask is not None:
    plt.figure(figsize=(4, 3))
    plt.imshow(mask.squeeze(), cmap='Blues', vmin=0, vmax=1)
    plt.title('Causal Mask (1 = can attend, 0 = blocked)')
    plt.xlabel('Key position')
    plt.ylabel('Query position')
    plt.colorbar()
    plt.tight_layout()
    plt.show()
else:
    print('Not yet implemented.')

### Exercise B2 — Verify causal attention

Run masked attention and confirm that:
1. The attention weight matrix is lower-triangular.
2. No future token receives nonzero weight.

In [ ]:
seq = 8
q = k = v = torch.randn(1, 1, seq, 32)   # batch=1, heads=1
mask = make_causal_mask(seq)

# TODO: call my_attention with the mask and check the upper triangle is ~0
if mask is not None:
    out, w = my_attention(q, k, v, mask=mask)
    if out is not None:
        w_np = w[0, 0].detach().numpy()
        print('Upper triangle sum (should be ~0):', np.triu(w_np, k=1).sum())

        plt.figure(figsize=(5, 4))
        plt.imshow(w_np, cmap='Blues', vmin=0, vmax=1)
        plt.title('Causal Attention Weights')
        plt.xlabel('Key pos'); plt.ylabel('Query pos')
        plt.colorbar()
        plt.tight_layout()
        plt.show()

## Part C — Using the Library Implementation

Compare your implementation against `src.models.attention.scaled_dot_product_attention`.

In [ ]:
from src.models.attention import scaled_dot_product_attention, create_causal_mask

B, H, S, dk = 2, 4, 10, 32
q = torch.randn(B, H, S, dk)
k = torch.randn(B, H, S, dk)
v = torch.randn(B, H, S, dk)

lib_out, lib_w = scaled_dot_product_attention(q, k, v)
print(f'Library output shape  : {lib_out.shape}')
print(f'Library weights shape : {lib_w.shape}')

# TODO: run your my_attention and check outputs are close
# my_out, my_w = my_attention(q, k, v)
# print('Outputs match:', torch.allclose(lib_out, my_out, atol=1e-5))

## Part D — Visualising Attention on Text

We'll use a simple character-level tokeniser and watch which characters attend to which.

In [ ]:
sentence = "the cat sat on the mat"
chars = list(sentence)
vocab = {c: i for i, c in enumerate(sorted(set(chars)))}
tokens = torch.tensor([vocab[c] for c in chars])

embed_dim = 16
embedding = nn.Embedding(len(vocab), embed_dim)
x = embedding(tokens).unsqueeze(0)   # (1, seq_len, embed_dim)

# Project to Q, K, V
Wq = nn.Linear(embed_dim, embed_dim, bias=False)
Wk = nn.Linear(embed_dim, embed_dim, bias=False)
Wv = nn.Linear(embed_dim, embed_dim, bias=False)

q = Wq(x).unsqueeze(1)  # (1, 1, seq_len, embed_dim) — single head
k = Wk(x).unsqueeze(1)
v = Wv(x).unsqueeze(1)

_, w = scaled_dot_product_attention(q, k, v)
w_np = w[0, 0].detach().numpy()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(w_np, cmap='Blues', vmin=0)
ax.set_xticks(range(len(chars))); ax.set_xticklabels(chars, fontfamily='monospace')
ax.set_yticks(range(len(chars))); ax.set_yticklabels(chars, fontfamily='monospace')
ax.set_xlabel('Key (attended to)')
ax.set_ylabel('Query (attending)')
ax.set_title(f'Attention weights — "{sentence}"')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Summary

- Attention computes **soft dictionary lookups** — every token can directly attend to every other.
- Scaling by $\sqrt{d_k}$ keeps logits in a range where softmax has healthy gradients.
- A **causal mask** enforces left-to-right structure for language generation.
- Even with random weights, the pattern emerges naturally from the QKV projections.

**Next:** `03_multihead_attention_starter.ipynb` — extend to multiple parallel attention heads.